# SPAR — 4 composites, gross/net toggle, multi-tile, as-of 0Q

Fabric **Python** notebook (not PySpark). Pulls SPAR Engine results for the four active
Longs Peak GIPS composites against each one's benchmark, for one or more SPAR components
("tiles"), on a gross / net / both basis, and lands them in `hbcm_datahub`.

```
grid = TILES x STRATEGIES x FEE_BASIS   ->   one SPAR calculation unit each
```

## Three things you must fill in before first run

| # | What | Where to get it | Why it can't be guessed |
|---|---|---|---|
| 1 | `TILES` component ids | Workstation SPAR document, or the discovery cell below | SPAR components are workstation-authored; the API can't create them |
| 2 | `BENCHMARKS` id + prefix per strategy | Workstation, or `BenchmarksApi` | **A wrong prefix returns a bare 400 with no useful message** |
| 3 | `ACCOUNT_PREFIX` | `AccountsApi` discovery cell | Depends on which `$$Performance.ofdb` the QR series live in |

Run **Cell 4 (discovery)** once interactively, paste the real values into Cell 3, then
never run discovery again — it's a lookup aid, not part of the pipeline.

## Versions — latest as of 2026-08-06

| Package | Version | Notes |
|---|---|---|
| `fds.sdk.SPAREngine` | **3.0.0** | released 2026-05-20 |
| `fds.sdk.utils` | 3.0.1 | OAuth helper |
| `fds.protobuf.stach.extensions` | 1.3.3 | STACH parsing |
| `deltalake` | 1.6.2 | OneLake write |

SPAREngine 3.0.0 is a **major bump for runtime hygiene, not an API redesign**. Per
FactSet's `BREAKING.md` (2026-05-20, "Python SDKs: Minimum Python Version Bump and
Dependency Updates") every Python SDK was bumped together: Python 3.7/3.8/3.9 support
dropped, and vulnerable dependencies updated — `urllib3` moved from `>=1.25.3,<2.1.0` to
`>=2.7.0`. Every model and method this notebook touches is unchanged between the 2.x
docs and 3.0.0.

Minimum Python is now **3.10**. Fabric Python notebook kernels are 3.10 / 3.11 / 3.12
with **3.12 the default**, so any current kernel satisfies it — prefer 3.12, since 3.10
reaches end of support in October 2026.

> ⚠️ The SPAR SDK vendored under `code/python/SPAREngine/v3/` **in this repo is 2.0.3**
> (last synced 2025-07-21) and predates both the `universeid` field and
> `SPARPeerUniverseApi`. Verify any SPAR question against upstream `main`, not the local
> mirror. PA Engine is at **4.0.0** upstream and carries a second, separate breaking
> change (2026-07-21: required fields dropped from `PADateParameters`) that matters for
> the PA side of the pipeline but not for this notebook.

## Library setup — read this before scheduling

Do **not** rely on `%pip install` here. Per Microsoft, inline installs are *disabled by
default in pipeline runs* and *unsupported in reference runs*, and `%pip`-installed
libraries are not retained across runs. Attach the four packages above to a **Fabric
Environment** and bind this notebook to it.

For interactive first-run only:
```
%pip install fds.sdk.SPAREngine==3.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
# HBCM_Config defines FACTSET_USER and FACTSET_APIKEY as plain strings.
# %run works in both interactive and pipeline mode; notebookutils.notebook.run() does not
# propagate variables, so keep this as %run.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.SPAREngine
from fds.sdk.SPAREngine.api import (
    spar_calculations_api,
    spar_peer_universe_api,   # SDK >= 2.1; absent from the 2.0.3 copy vendored here
    accounts_api,
    components_api,
    benchmarks_api,
)
from fds.sdk.SPAREngine.models import (
    SPARCalculationParametersRoot,
    SPARCalculationParameters,
    SPARIdentifier,
    SPARDateParameters,
    CalculationMeta,
)
from urllib3 import Retry   # SDK 3.0.0 requires urllib3 >= 2.7.0

# Fail loudly on a stale Environment rather than 400-ing later on universeid.
# Compare the major as an int — a string compare would rank "10.0.0" below "3.0.0".
SDK_VERSION = fds.sdk.SPAREngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 3, (
    f"fds.sdk.SPAREngine {SDK_VERSION} found; this notebook targets >=3.0.0. "
    "Check the bound Fabric Environment."
)
print("SPAREngine SDK", SDK_VERSION)

configuration = fds.sdk.SPAREngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
# Preferred once an app-config.json exists in Key Vault:
#   from fds.sdk.utils.authentication import ConfidentialClient
#   configuration = fds.sdk.SPAREngine.Configuration(
#       fds_oauth_client=ConfidentialClient(str(config_path)))

# Retry server-side failures only. Never add 429 with a naive backoff — urllib3 already
# honours Retry-After for 429, and 4xx other than 429 will not fix themselves.
configuration.retries = Retry(
    total=3,
    status_forcelist=[500, 502, 503, 504],
    backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.SPAREngine.ApiClient(configuration)
calc_api = spar_calculations_api.SPARCalculationsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK — the only cell you normally edit ============

# --- as-of date ------------------------------------------------------------
# "0Q" = most recent quarter end in FactSet relative-date grammar.
# SPAR Engine has no DatesApi, so relative tokens cannot be resolved to absolute
# dates via the SDK — they are validated server-side at calculation time.
# If "0Q" 400s, fall back to AS_OF_ABS below (computed, no API needed).
AS_OF = "0Q"
LOOKBACK = "-3Y"            # startdate; "-1Y", "-5Y", or an absolute YYYYMMDD
FREQUENCY = "Monthly"       # must be supported by the underlying return streams
CURRENCY = "USD"            # all GIPS series are USD-denominated
USE_EACH_PORTFOLIO_INCEPTION = False   # True for a since-inception comparison

def _prior_quarter_end(today=None):
    """Absolute fallback for AS_OF. Most recent completed calendar quarter end."""
    d = today or dt.date.today()
    q_start_month = ((d.month - 1) // 3) * 3 + 1
    qe = dt.date(d.year, q_start_month, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()

# --- fee basis toggle ------------------------------------------------------
# HBCM's gross and net streams are SEPARATE SYMBOLS in $$Performance.ofdb
# (GIPS_<STRATEGY>_G / _N), written by the QR Upload Returns Tool. So the toggle is a
# symbol-suffix switch, NOT the SPARIdentifier.returntype field.
#
# returntype is for accounts where FactSet computes net from a loaded fee schedule.
# Cell 4 prints the return types the API actually reports for these paths — if it comes
# back with real Gross/Net entries, switch to RETURNTYPE_MODE below and drop the suffix.
FEE_BASIS = "both"          # "gross" | "net" | "both"
SUFFIX = {"gross": "_G", "net": "_N"}
RETURNTYPE_MODE = False     # True => one symbol per strategy + returntype per basis
ACCOUNT_RETURNTYPE = {"gross": None, "net": None}   # fill from Cell 4 if RETURNTYPE_MODE

# --- the four composites ---------------------------------------------------
# Active QR upload scope. All Cap Core / International ADR / Small-Micro exist at the
# firm but are NOT in scope — do not add them without asking.
STRATEGIES = {
    "conc": {"label": "Concentrated Equity", "symbol_stem": "GIPS_CONC"},
    "lc":   {"label": "Large Cap",           "symbol_stem": "GIPS_LC"},
    "lcs":  {"label": "Large Cap Select",    "symbol_stem": "GIPS_LCS"},
    "smid": {"label": "SMID",               "symbol_stem": "GIPS_SMID"},
}

# TODO(1) — where the QR series live. Confirm with Cell 4.
ACCOUNT_PREFIX = "CLIENT:"

# TODO(2) — per-strategy benchmark. SPAR takes a SINGLE scalar benchmark per unit,
# which is why each strategy gets its own unit rather than sharing one.
# A wrong prefix yields a 400 with no diagnostic. Verify each with BenchmarksApi.
BENCHMARKS = {
    "conc": {"id": "<TODO>", "prefix": "BENCH:",   "returntype": None},
    "lc":   {"id": "<TODO>", "prefix": "BENCH:",   "returntype": None},
    "lcs":  {"id": "<TODO>", "prefix": "BENCH:",   "returntype": None},
    "smid": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
}

# TODO(3) — one entry per tile. A "tile" is a saved SPAR component; it fixes the
# statistic columns server-side. POST body values act as a one-time override over what
# the component has saved.
TILES = {
    "trailing_returns": "<TODO-32-char-component-id>",
    "risk_stats_3y":    "<TODO-32-char-component-id>",
    "calendar_years":   "<TODO-32-char-component-id>",
}

SPAR_DOCUMENT = "Client:/SPAR/HBCM"    # for the component discovery cell only

# --- peer universe (optional) ----------------------------------------------
# `universeid` is a real field on SPARCalculationParameters as of SDK 2.1+/3.0.0,
# fetched from /analytics/engines/spar/v3/peer-universe. This is the one thing the
# Power BI connector cannot do — it consumes whatever universe the saved component was
# configured with, whereas here it becomes a runtime parameter. Relevant for SMID's
# custom .USWEB universe.
#
# Leave a strategy as None to inherit the component's saved universe.
PEER_UNIVERSE = {"conc": None, "lc": None, "lcs": None, "smid": None}
PEER_UNIVERSE_CATEGORY = "Custom"      # for the discovery cell; e.g. "Custom", "eVestment"

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub (schema-enabled)
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
TABLE_PATH = f"{ONELAKE}/Tables/factset/spar_composite_returns"
RAW_DIR = f"{ONELAKE}/Files/raw/spar"

BASES = ["gross", "net"] if FEE_BASIS == "both" else [FEE_BASIS]
print(f"{len(TILES)} tiles x {len(STRATEGIES)} strategies x {len(BASES)} basis "
      f"= {len(TILES) * len(STRATEGIES) * len(BASES)} units")

In [ ]:
# === Cell 4: DISCOVERY — run interactively once, then leave it alone =======
# Resolves the TODOs above. Not part of the scheduled path.

acct_api = accounts_api.AccountsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)
bmk_api = benchmarks_api.BenchmarksApi(api_client)
peer_api = spar_peer_universe_api.SPARPeerUniverseApi(api_client)

# (a) What return types does SPAR actually report for these accounts?
#     This is what settles the gross/net question. ReturnType carries (name, id) —
#     the id is what goes in SPARIdentifier.returntype, and it is often a numeric code,
#     not the literal string "Net".
for code, meta in STRATEGIES.items():
    for basis in BASES:
        path = f"{meta['symbol_stem']}{SUFFIX[basis]}.ACCT"
        try:
            resp = acct_api.get_spar_returns_type(path)   # URL-encoded account path
            types = [(rt.get("name"), rt.get("id")) for rt in (resp.data.returns_type or [])]
            print(f"OK   {path:32s} -> {types}")
        except fds.sdk.SPAREngine.ApiException as e:
            print(f"FAIL {path:32s} -> {e.status} {e.body}")

# (b) Component ids available in the SPAR document -> TILES
try:
    comps = comp_api.get_spar_components(document=SPAR_DOCUMENT)
    for cid, c in (comps.data or {}).items():
        print(f"component {cid}  {getattr(c, 'name', '')}  ({getattr(c, 'category', '')})")
except fds.sdk.SPAREngine.ApiException as e:
    print(f"components lookup failed: {e.status} {e.body}")

# (c) Peer universes -> PEER_UNIVERSE. `category` is required; `name` and `directory`
#     are optional filters. Look for SMID's custom .USWEB universe here.
try:
    peers = peer_api.get_list_of_peer_universe(PEER_UNIVERSE_CATEGORY)
    print(peers)
except fds.sdk.SPAREngine.ApiException as e:
    print(f"peer universe lookup failed: {e.status} {e.body}")

# (d) Confirm a benchmark id resolves, and see its valid prefixes -> BENCHMARKS
#   print(bmk_api.get_spar_benchmark_by_id(id="RUSSELL:R2500"))

In [ ]:
# === Cell 5: build the calculation units ==================================
# One unit per (tile, strategy, basis).
#
# Why not collapse the four composites into a single unit's `accounts` list? Because
# SPARCalculationParameters.benchmark is a SCALAR, and the four composites have four
# different benchmarks. Accounts can only share a unit if they share a benchmark.
# (If two ever do share one, collapsing them is a valid optimisation — but the flat
# shape below keeps one unit key == one result block, which is far easier to debug.)

def account_identifier(code: str, basis: str) -> SPARIdentifier:
    stem = STRATEGIES[code]["symbol_stem"]
    if RETURNTYPE_MODE:
        return SPARIdentifier(
            id=stem,
            returntype=ACCOUNT_RETURNTYPE[basis],
            prefix=ACCOUNT_PREFIX,
        )
    return SPARIdentifier(
        id=f"{stem}{SUFFIX[basis]}",
        returntype=None,          # the symbol itself already is gross or net
        prefix=ACCOUNT_PREFIX,
    )

def build_unit(tile_id: str, code: str, basis: str) -> SPARCalculationParameters:
    bmk = BENCHMARKS[code]
    kwargs = dict(
        componentid=tile_id,
        accounts=[account_identifier(code, basis)],
        benchmark=SPARIdentifier(
            id=bmk["id"], returntype=bmk["returntype"], prefix=bmk["prefix"],
        ),
        dates=SPARDateParameters(
            startdate=LOOKBACK,
            enddate=AS_OF,
            frequency=FREQUENCY,
            useeachportfolioinception=USE_EACH_PORTFOLIO_INCEPTION,
        ),
        currencyisocode=CURRENCY,
    )
    # Only send universeid when set — passing None would override the component's
    # saved universe with nothing.
    if PEER_UNIVERSE.get(code):
        kwargs["universeid"] = PEER_UNIVERSE[code]
    return SPARCalculationParameters(**kwargs)

UNIT_KEYS = {}          # unit_key -> (tile_name, strategy_code, basis)
units = {}
for tile_name, tile_id in TILES.items():
    for code in STRATEGIES:
        for basis in BASES:
            key = f"{tile_name}__{code}__{basis}"
            units[key] = build_unit(tile_id, code, basis)
            UNIT_KEYS[key] = (tile_name, code, basis)

params_root = SPARCalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
print(f"built {len(units)} units")

In [ ]:
# === Cell 6: submit + poll ================================================
# Multi-unit calculations ALWAYS return 202 regardless of the deadline header, so the
# polling loop is the normal path here, not the exception.

def run_spar(params_root, deadline=10, poll_interval=3, timeout=900):
    """Returns (calc_id, list[(unit_id, result_or_None, status)], request_key)."""
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        spar_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res = calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id)
        out.append((unit_id, res, st))
    return calc_id, out

calc_id, results = run_spar(params_root)

failed = [(u, s) for u, r, s in results if r is None]
print(f"calc_id={calc_id}  ok={len(results) - len(failed)}  failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s}")
# Log calc_id alongside X-DataDirect-Request-Key for any FactSet support ticket —
# use the *_with_http_info variants when you need the response headers.
assert not failed, "some units failed; resolve before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing (write-once audit copy) ==========================
# FactSet earns a raw layer: calls are slow and async, STACH reshaping is fiddly, and
# vendor restatements mean the same call replayed later does NOT return what it
# originally returned. That matters for GIPS and Marketing Rule substantiation.
# Files, not Delta — raw stays invisible to the SQL endpoint, which is correct.

asof_tag = AS_OF_ABS   # label raw by resolved quarter end, never by a relative token
for unit_id, res, _ in results:
    notebookutils.fs.put(
        f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
        json.dumps(res.to_dict(), default=str),
        True,
    )
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

In [ ]:
# === Cell 8: STACH -> tidy DataFrame ======================================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

def stach_to_dataframes(api_response):
    """STACH v2 payload -> list[DataFrame], one per table."""
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    tables = ext.convert(json.dumps(api_response.to_dict(), default=str))
    return [pd.DataFrame(t.data, columns=t.columns) for t in tables]

frames = []
for unit_id, res, _ in results:
    tile_name, code, basis = UNIT_KEYS[unit_id]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        df.insert(0, "asof_date", asof_tag)
        df.insert(1, "tile", tile_name)
        df.insert(2, "strategy_code", code)
        df.insert(3, "strategy", STRATEGIES[code]["label"])
        df.insert(4, "fee_basis", basis)
        df.insert(5, "benchmark_id", BENCHMARKS[code]["id"])
        df.insert(6, "table_ix", i)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
tidy.columns = [str(c).strip().replace(" ", "_").lower() for c in tidy.columns]
tidy = tidy.astype({c: "string" for c in tidy.select_dtypes("object").columns})
print(tidy.shape)
display(tidy.head(20))

In [ ]:
# === Cell 9: idempotent write to hbcm_datahub =============================
# delete-then-append on asof_date, so a pipeline retry or manual re-run replaces the
# snapshot instead of doubling it. mode="append" alone would silently duplicate.
from deltalake import DeltaTable, write_deltalake

try:
    DeltaTable(TABLE_PATH).delete(f"asof_date = '{asof_tag}'")
    mode = "append"
except Exception:
    mode = "overwrite"          # table does not exist yet

write_deltalake(TABLE_PATH, tidy, mode=mode, schema_mode="merge")
print(f"wrote {len(tidy)} rows to factset.spar_composite_returns (mode={mode})")

# No partitioning: this table is kilobytes. Partitioning a small table costs more in
# metadata than it saves in pruning.
# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a
# framed model still points at gives users query errors on missing files.
# Order is always: write -> frame -> vacuum.

## Validation checklist

- [ ] Gross figures exceed net for every strategy and period. If gross == net, the
      suffix switch isn't reaching a distinct symbol — check `RETURNTYPE_MODE`.
- [ ] `asof_date` matches the quarter end you expect. `AS_OF = "0Q"` is resolved
      server-side; `AS_OF_ABS` is what gets written. If those disagree the labels lie.
- [ ] Returns tie to the composite performance report — that report, not SPAR, is the
      GIPS authority.
- [ ] Unit count == tiles x 4 x basis, and none failed.
- [ ] Cell 2's version assert passed, so the Environment really is on SDK >= 3.0.0.

## Known sharp edges

| Symptom | Cause |
|---|---|
| 400, no detail | Wrong `prefix` on account or benchmark. The single most common failure. |
| 400 on dates | `"0Q"` unsupported for this component/frequency — use `AS_OF_ABS`. |
| 400 mentioning `universeid` | Environment is on an SDK older than 2.1 — the field didn't exist. Cell 2's assert should catch this first. |
| `ImportError` on `spar_peer_universe_api` | Same cause: stale SDK. |
| 404 fetching a result | Calculation id expired (TTL is hours). Resubmit. |
| 429 | Concurrency limit, typically 5–10 concurrent calcs per user. Reduce tiles per run. |
| Empty tables for one tile | The component's saved date range or peer universe conflicts with the override. |
| Works interactively, fails in pipeline | `%pip` install — bind a Fabric Environment instead. |

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (SPAREngine v3, SDK 3.0.0)
— `SPARCalculationsApi.md`, `SPARCalculationParameters.md`, `SPARIdentifier.md`,
`SPARDateParameters.md`, `CalculationMeta.md`, `AccountsApi.md`
(`get_spar_returns_type`), `SPARPeerUniverseApi.md`, `ReturnType.md`, plus `BREAKING.md`
for the 2026-05-20 Python-SDK bump.

Not against `code/python/SPAREngine/v3/` in this repo, which is pinned at 2.0.3
(2025-07-21) and lacks `universeid` and `SPARPeerUniverseApi`.

Microsoft Learn — [notebook limitations](https://learn.microsoft.com/fabric/data-engineering/notebook-limitation),
[%run](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook#run-notebooks),
[Python kernel lifecycle](https://learn.microsoft.com/fabric/data-engineering/python-notebook-runtime-lifecycle),
[pandas to lakehouse](https://learn.microsoft.com/fabric/data-engineering/lakehouse-notebook-load-data#load-data-with-pandas-api).